In [1]:
import pandas as pd

results = pd.read_csv("../data/raw/international_results/results.csv")
shootouts = pd.read_csv("../data/raw/international_results/shootouts.csv")

print(results.shape)
print(results.dtypes)
results.head()

(49477, 9)
date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
print(f"Manquants :\n{results.isnull().sum()}")

results["date"] = pd.to_datetime(results["date"])
print(f"\n\nDe {results["date"].min()} à {results["date"].max()}")

print(f"\n\nNombre de tournois uniques : {results["tournament"].nunique()}")
print(results["tournament"].value_counts().head(20))

all_teams = pd.concat([results["home_team"], results["away_team"]]).unique()
print(f"\n\nNombre d'équipes uniques : {len(all_teams)}")

print(f"\n\nStats buts domicile :\n{results['home_score'].describe()}")
print(f"\nStats buts extérieur :\n{results['away_score'].describe()}")

Manquants :
date           0
home_team      0
away_team      0
home_score    60
away_score    60
tournament     0
city           0
country        0
neutral        0
dtype: int64


De 1872-11-30 00:00:00 à 2026-06-27 00:00:00


Nombre de tournois uniques : 200
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                               

In [3]:
print(results[results["home_score"] == 31])

wc2026 = results[results["tournament"] == "FIFA World Cup"]
print(wc2026[wc2026["date"] >= "2026-01-01"])

results["decade"] = (results["date"].dt.year // 10) * 10
print(results.groupby("decade").size())

            date  home_team       away_team  home_score  away_score  \
25425 2001-04-11  Australia  American Samoa        31.0         0.0   

                         tournament           city    country  neutral  
25425  FIFA World Cup qualification  Coffs Harbour  Australia    False  
            date      home_team               away_team  home_score  \
49405 2026-06-11         Mexico            South Africa         2.0   
49406 2026-06-11    South Korea          Czech Republic         2.0   
49407 2026-06-12         Canada  Bosnia and Herzegovina         1.0   
49408 2026-06-12  United States                Paraguay         4.0   
49409 2026-06-13          Qatar             Switzerland         1.0   
...          ...            ...                     ...         ...   
49472 2026-06-27         Jordan               Argentina         NaN   
49473 2026-06-27       Colombia                Portugal         NaN   
49474 2026-06-27       DR Congo              Uzbekistan         NaN   
4

In [4]:
TOURNAMENTS_TO_KEEP = [
    "FIFA World Cup",
    "FIFA World Cup qualification",
    "UEFA Euro",
    "UEFA Euro qualification",
    "UEFA Nations League",
    "Copa América",
    "African Cup of Nations",
    "African Cup of Nations qualification",
    "AFC Asian Cup",
    "AFC Asian Cup qualification",
    "Gold Cup",
    "CONCACAF Nations League",
    "Friendly",
]

# Appliquer tous les filtres
df = results.copy()
df["date"] = pd.to_datetime(df["date"])

# 1. Cutoff 1993 (FIFA ranking era)
df = df[df["date"] >= "1993-01-01"]

# 2. Garder uniquement les tournois pertinents
df = df[df["tournament"].isin(TOURNAMENTS_TO_KEEP)]

# 3. Supprimer les NaN de scores (élimine automatiquement les matchs futurs, dont CdM 2026 à venir)
df = df.dropna(subset=["home_score", "away_score"])

# Résultat
print(f"Shape final : {df.shape}")
print(f"De {df['date'].min()} à {df['date'].max()}")
print(df["tournament"].value_counts())

Shape final : (24925, 10)
De 1993-01-01 00:00:00 à 2026-06-14 00:00:00
tournament
Friendly                                10211
FIFA World Cup qualification             6821
UEFA Euro qualification                  1991
African Cup of Nations qualification     1745
UEFA Nations League                       658
AFC Asian Cup qualification               613
African Cup of Nations                    606
FIFA World Cup                            512
CONCACAF Nations League                   422
Gold Cup                                  404
Copa América                              352
UEFA Euro                                 308
AFC Asian Cup                             282
Name: count, dtype: int64


In [5]:
df = df.drop(columns=["decade"], errors="ignore")

df.to_csv("../data/processed/matches_clean.csv", index=False)
print("Sauvegardé : data/processed/matches_clean.csv")

Sauvegardé : data/processed/matches_clean.csv


In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/features.csv")

feature_cols = [
    "home_form_scored", "home_form_conceded", "home_form_win_rate",
    "away_form_scored", "away_form_conceded", "away_form_win_rate",
    "h2h_avg_total_goals", "h2h_home_win_rate",
]

total = len(df)
for col in feature_cols:
    n_nan = df[col].isna().sum()
    print(f"{col:30s} → {n_nan:5d} NaN  ({n_nan/total*100:.1f}%)")

home_form_scored               →   138 NaN  (0.6%)
home_form_conceded             →   138 NaN  (0.6%)
home_form_win_rate             →   138 NaN  (0.6%)
away_form_scored               →   146 NaN  (0.6%)
away_form_conceded             →   146 NaN  (0.6%)
away_form_win_rate             →   146 NaN  (0.6%)
h2h_avg_total_goals            →  5909 NaN  (23.7%)
h2h_home_win_rate              →  5909 NaN  (23.7%)


In [7]:
import requests
import io
import pandas as pd

base = "https://www.eloratings.net/"

for filename in ["teams.tsv", "en.teams.tsv", "menu.tsv"]:
    resp = requests.get(
        base + filename,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=15
    )
    print(f"\n=== {filename} (status {resp.status_code}) ===")
    if resp.status_code == 200:
        print(resp.text[:800])
    else:
        print("Echec")


=== teams.tsv (status 200) ===
DY	YE
RV	VN
CS	CZ
SU	RU
DS	RU
WG	DE
CQ	CD
ZR	CD
HA	CZ
EM	CZ
DI	ID
AN	CW
RH	ZW
ZD	ZW
ND	ZM
BU	MM
NH	VU
AY	MY
KU	KH
UM	SR
DH	BJ
UV	BF
CE	LK
GC	GH
RM	RS
YU	RS
GB	EN
RI	IN
RG	GY
TY	TZ
BC	CD
RR	EG
CJ	CD
FS	DJ
UK	EN
PU	GW
RB	BI
FT	TG
NN	MW
UH	CF
FC	CG
MI	ML
BX	BZ
DN	YE
KT	KN
WM	WS
AH	AT
SZ	SW
FN	ML
MK	NM
KA	KR
CB	CG
EP	BD
OD	MD
OR	BY
UT	LS

=== en.teams.tsv (status 200) ===
DN	Aden
AF	Afghanistan
AL	Albania
DZ	Algeria
AD	Andorra
AO	Angola
AI	Anguilla
AG	Antigua and Barbuda	Antigua & Barbuda	Antigua/Barbuda	Antigua/Barb
AR	Argentina
AM	Armenia
AW	Aruba
AU	Australia
AT	Austria
AH	Austria-Hungary
AZ	Azerbaijan
BS	Bahamas
BS_loc	in the Bahamas	in Bahamas
BH	Bahrain
BD	Bangladesh
BB	Barbados
UT	Basutoland
BY	Belarus
BC	Belgian Congo	Belg Congo
BE	Belgium
BZ	Belize
OR	Belorussia
BJ	Benin
BM	Bermuda
BT	Bhutan
HA	Bohemia
EM	Bohemia and Moravia	Bohemia & Moravia	Bohemia/Moravia	Bohemia/Morav	Bohemia/Mor
BO	Bolivia
BQ	Bonaire
BA	Bosnia and Herzegovina	Bosnia & Herzegovi

In [8]:
import requests

base = "https://www.eloratings.net/"

candidates = [
    "World.tsv",
    "WR.tsv", 
    "Argentina.tsv",
    "France.tsv",
]

for f in candidates:
    resp = requests.get(base + f, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    print(f"\n=== {f} (status {resp.status_code}) ===")
    if resp.status_code == 200:
        print(resp.text[:400])


=== World.tsv (status 200) ===
1	1	ES	2129	1	2189	7	1946	19	1805	0	â43	0	â43	0	â28	+2	+71	+5	+113	+2	+128	783	341	302	140	462	138	183	1595	699
2	2	AR	2115	1	2172	5	1987	26	1751	0	+2	0	+2	0	â17	â1	â30	+6	+142	â1	+32	1111	381	419	311	612	228	271	2117	1136
3	3	FR	2084	1	2135	16	1795	40	1514	0	+22	0	+22	0	+29	â1	+8	0	â24	+1	+99	941	475	341	125	476	270	195	1713	1276
4	4	EN	2024	1	2213	4	1983	17	1810	0	â18	0	â18

=== WR.tsv (status 404) ===

=== Argentina.tsv (status 200) ===
1902	07	20	UY	AR	0	6	F		-21	1879	2021	â	â	4	1
1903	09	13	AR	UY	2	3	F		-16	2005	1895	0	+1	1	6
1905	08	15	AR	UY	0	0	LPT		-8	1997	1903	0	+1	1	5
1906	07	09	AR	ZA	0	1	F		-17	1980	1817	0	â	1	10
1906	08	15	UY	AR	0	2	LPT		-22	1851	2002	0	0	7	1
1906	10	21	AR	UY	2	1	NWT		6	2008	1845	0	â1	1	8
1907	08	15	AR	UY	2	1	LPT		5	2013	1840	0	0	1	8
1907	10	06	UY	AR	1	2	NWT		-12	1828	2025	0	0	8	1
1908	08	15	UY	A

=== France.tsv (status 200) ===
1904	05	01	BE	FR	3	3	ECT		-10	1790	1610	â	â	8	9
1905	02	12	FR	CH

In [1]:
import pandas as pd

matches = pd.read_csv("../data/processed/matches_clean.csv")
codes   = pd.read_csv("../data/external/elo_team_codes.csv")

# Noms uniques de chaque côté
matches_teams = set(pd.concat([matches["home_team"], matches["away_team"]]).unique())
elo_teams     = set(codes["name"].unique())

# Équipes présentes dans matches_clean mais absentes du référentiel elo
missing = matches_teams - elo_teams
print(f"Équipes dans matches_clean sans correspondance directe : {len(missing)}")
print(sorted(missing))

Équipes dans matches_clean sans correspondance directe : 62
['Abkhazia', 'Alderney', 'Ambazonia', 'American Samoa', 'Andalusia', 'Artsakh', 'Barawa', 'Basque Country', 'Brittany', 'Canary Islands', 'Cascadia', 'Catalonia', 'Chechnya', 'Cilento', 'Corsica', 'Curaçao', 'Czech Republic', 'Donetsk PR', 'East Turkestan', 'Elba Island', 'Ellan Vannin', 'Franconia', 'Galicia', 'Gozo', 'Guernsey', 'Jersey', 'Kernow', 'Luhansk PR', 'Macau', 'Madrid', 'Micronesia', 'Occitania', 'Padania', 'Panjab', 'Parishes of Jersey', 'Provence', 'Raetia', 'Republic of Ireland', 'Romani people', 'Ryūkyū', 'Réunion', 'Saint Barthélemy', 'Saint Helena', 'Saugeais', 'Sealand', 'Seborga', 'Silesia', 'South Ossetia', 'Surrey', 'Székely Land', 'Sápmi', 'São Tomé and Príncipe', 'Tamil Eelam', 'Ticino', 'Timor-Leste', 'United Koreans in Japan', 'United States Virgin Islands', 'Vatican City', 'West Papua', 'Ynys Môn', 'Yorkshire', 'Åland Islands']


In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])

# Équipes avec elo manquant côté domicile
missing_home = df[df["home_elo"].isna()]
missing_away = df[df["away_elo"].isna()]

print("Équipes home avec elo manquant :")
print(missing_home["home_team"].value_counts())

print("\nÉquipes away avec elo manquant :")
print(missing_away["away_team"].value_counts())

print("\nDates des matchs avec elo manquant (home) :")
print(missing_home["date"].describe())

Équipes home avec elo manquant :
home_team
Samoa    1
Name: count, dtype: int64

Équipes away avec elo manquant :
away_team
Samoa     1
Monaco    1
Name: count, dtype: int64

Dates des matchs avec elo manquant (home) :
count                      1
mean     1996-11-13 00:00:00
min      1996-11-13 00:00:00
25%      1996-11-13 00:00:00
50%      1996-11-13 00:00:00
75%      1996-11-13 00:00:00
max      1996-11-13 00:00:00
Name: date, dtype: object


In [1]:
import math
mu = 0.6376358866691589
p_away_scores_at_least_1 = 1 - math.exp(-mu)
print(p_away_scores_at_least_1)

0.4714595222190644
